In [1]:
#!/usr/bin/env python3
# Copyright (c) 2024 Arista Networks, Inc.  All rights reserved.
# Arista Networks, Inc. Confidential and Proprietary.

# Jupyter Notebook for visualizing ODS data from Meta.

# Note that this is based on the data file format shared in April 2024
# which may change in the future. At the time, the format was a CSV
# with no headers and the following columns:
# sensor name (host::sensor),value,timestamp,some numeric value I can't identify

# Instructions:
# To use, simply copy your ODS CSV to your local machine then update the
# odsCsvFile value to point to your file. Then run the notebook and an
# interactive time-series graph will be produced below.

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import sys
import tempfile

In [3]:
# ADD YOUR ODS FILE HERE
odsCsvFile = 'rsw021.p001.f01.eag6.csv'

In [4]:
def addHeaderToCsvFile(csvFile):
   # ODS data dump CSV doesn't have headers so create a tempfile with the correct
   # headers to use in the rest of the script.
   tempCsv = tempfile.NamedTemporaryFile(prefix='MetaOdsViewer')
   tempCsv.write(b'name,value,timestamp,id\n')
   with open(csvFile) as csvf:
      # Go through line-by-line to avoid reading large CSV into memory.
      for line in csvf:
         tempCsv.write(bytes(line, 'utf-8'))
   return tempCsv

In [5]:
def getOdsDataFrame(csvFile):
   # Given a CSV file of ODS data, create a pandas dataframe with the correct indices.
   tempCsvFileWithHeader = addHeaderToCsvFile(csvFile)
   odsData = pd.read_csv(tempCsvFileWithHeader.name)
   # Transform the dataframe as follows:
   # 1. Convert timestamp string to pd.datetime format.
   # 2. Unstack sensor names such that every sensor has a dedicated column.
   odsData['timestamp'] = odsData['timestamp'].apply(pd.to_datetime)
   odsData['name'] = odsData['name'].str.split('::').str.get(1)
   odsData = (odsData.set_index(['timestamp', 'name'])['value'].
                  unstack('name').reset_index())
   return odsData

In [6]:
odsDataFrame = getOdsDataFrame(odsCsvFile)

In [7]:
fig = px.line(odsDataFrame, x='timestamp', y=odsDataFrame.columns)
fig.show()